In [1]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/naver/disk2/HoangDPB/rPPG-Toolbox"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.PhysNet import PhysNet_padding_Encoder_Decoder_MAX

In [2]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "raw_data/PPG_Dataset")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/groupC")
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/groupC/PPG_Dataset")

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 25       # PPG sensor sampling rate (Hz)

# ----- PhysNet preprocessing params -----
CHUNK_LENGTH = 128     # frames per clip
IMG_H, IMG_W = 72, 72  # resolution
LABEL_TYPE   = "DiffNormalized"  # cumsum in post-processing
DATA_FORMAT  = "NCDHW"

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Device: cuda:0
PREPROCESSED_PATH: /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupC
OUTPUT_DIR: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupC/PPG_Dataset


In [3]:
# Models list -- uncomment to run additional pretrained models
MODELS = [
    ("PURE_PhysNet",                   "Physnet", "final_model_release/PURE_PhysNet_DiffNormalized.pth"),
    ("SCAMPS_PhysNet",               "Physnet", "final_model_release/SCAMPS_PhysNet_DiffNormalized.pth"),
    ("UBFC-rPPG_PhysNet",            "Physnet", "final_model_release/UBFC-rPPG_PhysNet_DiffNormalized.pth"),
    ("BP4D_PseudoLabel_PhysNet",     "Physnet", "final_model_release/BP4D_PseudoLabel_PhysNet_DiffNormalized.pth"),
    ("MA-UBFC_physnet",              "Physnet", "final_model_release/MA-UBFC_physnet.pth"),
]

# Default to first model
MODEL_NAME, MODEL_TYPE, MODEL_REL_PATH = MODELS[0]
MODEL_PATH = os.path.join(REPO_ROOT, MODEL_REL_PATH)
print(f"Selected model: {MODEL_NAME}")
print(f"Model path: {MODEL_PATH}")

Selected model: PURE_PhysNet
Model path: /home/naver/disk2/HoangDPB/rPPG-Toolbox/final_model_release/PURE_PhysNet_DiffNormalized.pth


In [4]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [5]:
# Read and time-align PPG signal

def read_ppg_synced(session_path, num_frames, fps=30):
    """Read PPG green channel from ppg.csv and resample to video frame times.

    The PPG sensor timestamps and video start timestamp are both in ms
    Unix epoch stored in metadata.json (sync_markers.video_start).
    Resampling is done by linear interpolation onto frame-spaced timestamps.

    Returns:
        ppg (np.ndarray): shape (num_frames,), dtype float32, raw green values.
    """
    meta_path = os.path.join(session_path, "metadata.json")
    with open(meta_path, "r") as f:
        meta = json.load(f)
    video_start_ms = meta["sync_markers"].get("video_start")
    if video_start_ms is None:
        video_start_ms = meta.get("start_timestamp")
    if video_start_ms is None:
        raise ValueError(f"No video_start or start_timestamp found in {meta_path}")
    video_start_ms = float(video_start_ms)

    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_ts    = ppg_df["timestamp"].values.astype(np.float64)
    ppg_green = ppg_df["green"].values.astype(np.float64)

    # convert ppg timestamps to seconds relative to video start
    ppg_t = (ppg_ts - video_start_ms) / 1000.0

    # frame timestamps in seconds
    frame_t = np.arange(num_frames, dtype=np.float64) / fps

    # clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])

    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_green)
    return ppg_resampled.astype(np.float32)

In [6]:
# Normalization functions

def diff_normalize_data(data):
    """DiffNormalized: (frame[t+1]-frame[t]) / (frame[t+1]+frame[t]+1e-7), then / std."""
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [7]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [8]:
# Discover subjects + read device-measured HR from hr.csv

def read_gt_hr(session_path):
    """Return average HR from hr.csv (hrStatus==1, hr>0).

    hr.csv has IBI columns whose values are arrays like [720, 695],
    which contain commas and break the standard CSV parser.
    Reading only the first 5 columns avoids the issue.
    """
    hr_path = os.path.join(session_path, "hr.csv")
    df = pd.read_csv(
        hr_path,
        usecols=[0, 1, 2, 3, 4],
        names=["id", "dataReceived", "timestamp", "hr", "hrStatus"],
        header=0,
    )
    valid = df[(df["hrStatus"] == 1) & (df["hr"] > 0)]["hr"]
    if len(valid) == 0:
        return float("nan")
    return float(valid.mean())


# Discover subject folders directly
all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_dirs = glob.glob(os.path.join(RAW_DATA_PATH, subj_id, "*"))
    if not session_dirs:
        print(f"No session folder for {subj_id}, skipping.")
        continue
    session_path = session_dirs[0]

    session_name  = os.path.basename(session_path)
    video_pattern = os.path.join(RAW_DATA_PATH, "videos",
                                  f"{subj_id}_{session_name}.mp4")
    video_files = glob.glob(video_pattern)
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    gt_hr = read_gt_hr(session_path)

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
        "gt_hr":        gt_hr,
    })
    print(f"  {subj_id}  HR_device={gt_hr:.4f} bpm  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

Found 60 subject folders

  S_000  HR_device=83.5375 bpm  video=S_000_baseline_1768470386897.mp4
  S_001  HR_device=80.3974 bpm  video=S_001_baseline_1768465415430.mp4
  S_002  HR_device=122.4792 bpm  video=S_002_baseline_1768559029065.mp4
  S_003  HR_device=114.9216 bpm  video=S_003_baseline_1768561970957.mp4
  S_004  HR_device=79.1250 bpm  video=S_004_baseline_1768560982129.mp4
  S_005  HR_device=92.7875 bpm  video=S_005_baseline_1768558641873.mp4
  S_006  HR_device=106.7284 bpm  video=S_006_baseline_1768557426231.mp4
  S_007  HR_device=103.5309 bpm  video=S_007_baseline_1768560100207.mp4
  S_008  HR_device=90.2963 bpm  video=S_008_baseline_1768559192265.mp4
  S_009  HR_device=81.0886 bpm  video=S_009_baseline_1768032143238.mp4
  S_010  HR_device=98.0617 bpm  video=S_010_baseline_1768548590301.mp4
  S_011  HR_device=72.5679 bpm  video=S_011_baseline_1768560819006.mp4
  S_012  HR_device=81.4881 bpm  video=S_012_baseline_1768791488131.mp4
  S_013  HR_device=90.2449 bpm  video=S_013_bas

In [9]:
# Data preprocessing

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    ppg_signal = read_ppg_synced(session_path, T, fps=VIDEO_FPS)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Face crop and resize
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)

    # DiffNormalized data: 3 channels
    diff_data = diff_normalize_data(frames_cropped)

    # DiffNormalized label
    label = diff_normalize_label(ppg_signal)

    # Chunk into clips of CHUNK_LENGTH frames
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([diff_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])

    # Save per subject
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, data_clips[chunk_idx])   # (CHUNK_LENGTH, H, W, 3)
        np.save(label_path, label_clips[chunk_idx])  # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

Cleared and recreated: /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupC

=== Processing S000 ===
  Video: 2706 frames @ 30 fps
  PPG green: min=-286206, max=237092
  21 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupC/S000

=== Processing S001 ===
  Video: 2691 frames @ 30 fps
  PPG green: min=-23646, max=248396
  21 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupC/S001

=== Processing S002 ===
  Video: 2789 frames @ 30 fps
  PPG green: min=-44811, max=180257
  21 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupC/S002

=== Processing S003 ===
  Video: 2729 frames @ 30 fps
  PPG green: min=-104560, max=349890
  21 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupC/S003

=== Processing S004 ===
  Video: 2722 frames @ 30 fps
  PPG green: min=-300061, max=200804
  21 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupC/S004

=== Processing S005 ===
  Video: 

In [10]:
# PyTorch Dataset + DataLoader

class PhysNetDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (T,)

        # NDHWC -> NCDHW: transpose (3, 0, 1, 2)
        data = np.transpose(data, (3, 0, 1, 2))  # (3, T, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = PhysNetDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)
print(f"DataLoader ready: {len(loader)} batches")

Dataset: 1262 clips
DataLoader ready: 316 batches


In [11]:
# Post-processing helpers

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp(pred_chunks, label_chunks, fs=30, diff_flag=True):
    """Per-subject BVP post-processing with optional cumsum for DiffNormalized."""
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    if diff_flag:
        pred  = detrend(np.cumsum(pred),  100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred  = detrend(pred,  100)
        label = detrend(label, 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

In [12]:
# Blink rate detection

def detect_blink_rate(video_path, fps=30, large_box_coef=1.5):
    """Estimate blink rate (blinks/min) using eye-strip brightness analysis.

    Steps:
      1. Detect face on frame 0 with Haar Cascade.
      2. Define eye strip = rows [20%, 50%] of face bbox height.
      3. Compute mean grayscale brightness in eye strip per frame.
      4. Invert (blinks = dips) and bandpass filter [0.1, 0.9] Hz.
      5. Count peaks with minimum separation of 1 second.
    """
    frames = read_video_frames(video_path)
    T = len(frames)

    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(xml_path)
    gray0 = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray0, scaleFactor=1.3, minNeighbors=5)
    H, W = frames[0].shape[:2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H

    eye_top    = y + int(fh * 0.20)
    eye_bottom = y + int(fh * 0.50)
    eye_left   = x
    eye_right  = x + fw

    brightness = np.array([
        np.mean(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY)[eye_top:eye_bottom, eye_left:eye_right])
        for f in frames
    ])

    # blinks = dips in brightness -> invert for find_peaks
    brightness_inv = -brightness.astype(np.float64)

    # bandpass [0.1, 0.9] Hz = [6, 54] blinks/min
    b, a = signal.butter(2, [0.1 / fps * 2, 0.9 / fps * 2], btype="bandpass")
    filtered = signal.filtfilt(b, a, brightness_inv)

    # minimum 1 second between detected blinks
    min_dist = int(fps)
    peaks, _ = signal.find_peaks(filtered, distance=min_dist)

    duration_min = T / fps / 60.0
    return len(peaks) / duration_min


# Compute blink rate for each subject
blink_rate_dict = {}
for subj in subjects:
    subj_key   = subj["subj_key"]
    video_path = subj["video_path"]
    print(f"  {subj_key} ...", end=" ", flush=True)
    blink_rate = detect_blink_rate(video_path, fps=VIDEO_FPS)
    blink_rate_dict[subj_key] = blink_rate
    print(f"{blink_rate:.2f} blinks/min")

print("\nBlink rates:", {k: f"{v:.2f}" for k, v in blink_rate_dict.items()})

  S000 ... 27.94 blinks/min
  S001 ... 25.42 blinks/min
  S002 ... 30.98 blinks/min
  S003 ... 28.36 blinks/min
  S004 ... 29.10 blinks/min
  S005 ... 28.41 blinks/min
  S006 ... 24.44 blinks/min
  S007 ... 28.22 blinks/min
  S008 ... 26.90 blinks/min
  S009 ... 32.55 blinks/min
  S010 ... 27.69 blinks/min
  S011 ... 29.88 blinks/min
  S012 ... 27.80 blinks/min
  S013 ... 28.95 blinks/min
  S014 ... 31.78 blinks/min
  S015 ... 25.45 blinks/min
  S016 ... 31.85 blinks/min
  S017 ... 27.81 blinks/min
  S018 ... 26.26 blinks/min
  S019 ... 28.33 blinks/min
  S020 ... 25.31 blinks/min
  S021 ... 26.46 blinks/min
  S022 ... 31.44 blinks/min
  S023 ... 27.73 blinks/min
  S024 ... 24.95 blinks/min
  S025 ... 27.11 blinks/min
  S026 ... 31.95 blinks/min
  S027 ... 29.50 blinks/min
  S028 ... 27.68 blinks/min
  S029 ... 29.82 blinks/min
  S030 ... 28.94 blinks/min
  S031 ... 30.55 blinks/min
  S032 ... 27.30 blinks/min
  S033 ... 25.30 blinks/min
  S034 ... 28.17 blinks/min
  S035 ... 29.15 bli

In [13]:
# Inference loop -> per-subject results -> aggregate metrics -> export
# Results saved to results_groupC/{model_name}/

# lookup: subj_key -> subject info dict
subjects_by_key = {s["subj_key"]: s for s in subjects}

# Iterate over all models in the MODELS list
for model_name, model_type, model_rel_path in MODELS:
    model_path = os.path.join(REPO_ROOT, model_rel_path)
    print(f"\n{'='*70}")
    print(f"Model: {model_name}")
    print(f"Path:  {model_path}")
    print(f"{'='*70}")

    # Load model
    model = PhysNet_padding_Encoder_Decoder_MAX(frames=CHUNK_LENGTH)
    state_dict = torch.load(model_path, map_location=DEVICE)
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model loaded. Total parameters: {num_params:,}")

    # Run inference
    bvp_preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray}
    bvp_labels_dict = {}

    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            data, labels_batch, batch_subjects, batch_chunk_ids = batch

            # data shape: (N, 3, T, H, W) -- already NCDHW from dataset
            data = data.to(DEVICE)

            # PhysNet returns (rPPG, x_visual, x_visual3232, x_visual1616)
            rPPG, _, _, _ = model(data)

            # rPPG shape: (N, T)
            rPPG_np   = rPPG.cpu().numpy()
            labels_np = labels_batch.numpy()

            N = data.shape[0]
            for i in range(N):
                subj = batch_subjects[i]
                cid  = int(batch_chunk_ids[i])

                if subj not in bvp_preds_dict:
                    bvp_preds_dict[subj]  = {}
                    bvp_labels_dict[subj] = {}

                bvp_preds_dict[subj][cid]  = rPPG_np[i]
                bvp_labels_dict[subj][cid] = labels_np[i]

    print(f"\nInference complete. Subjects: {sorted(bvp_preds_dict.keys())}")

    # Per-subject results
    per_subject_results = []
    hr_preds_all = []
    gt_hrs_all   = []
    snr_all      = []

    FS = VIDEO_FPS

    print(f"\n{'Subject':<10} {'HR_pred':>10} {'HR_gt':>10} {'HR_err':>8} {'SNR':>7} {'Blinks':>7}")
    print("-" * 60)

    for subj_key in sorted(bvp_preds_dict.keys()):
        hr_pred, hr_label, _, pred_processed = process_bvp(
            bvp_preds_dict[subj_key], bvp_labels_dict[subj_key], fs=FS, diff_flag=True
        )

        gt_hr      = subjects_by_key[subj_key]["gt_hr"]
        snr_db     = calculate_snr(pred_processed, gt_hr, FS)
        blink_rate = blink_rate_dict.get(subj_key, float("nan"))
        hr_err     = hr_pred - gt_hr

        # map subj_key ("S000") back to original ID ("S_000")
        subj_id = subj_key[0] + "_" + subj_key[1:]

        per_subject_results.append({
            "name":                subj_id,
            "predicted_heartrate": hr_pred,
            "real_heartrate":      gt_hr,
            "heartrate_error":     hr_err,
            "blink_rate":          blink_rate,
            "snr_db":              snr_db,
        })

        hr_preds_all.append(hr_pred)
        gt_hrs_all.append(gt_hr)
        snr_all.append(snr_db)

        print(f"{subj_id:<10} {hr_pred:>10.3f} {gt_hr:>10.3f} {hr_err:>8.3f} {snr_db:>7.2f} {blink_rate:>7.2f}")

    hr_preds_all = np.array(hr_preds_all)
    gt_hrs_all   = np.array(gt_hrs_all)
    snr_all      = np.array(snr_all)

    # Aggregate metrics
    n = len(hr_preds_all)
    assert n > 0, "No subjects to evaluate."

    err   = hr_preds_all - gt_hrs_all
    abs_e = np.abs(err)
    sq_e  = err ** 2
    rel_e = abs_e / (np.abs(gt_hrs_all) + 1e-9)

    mae       = float(np.mean(abs_e))
    mae_se    = float(np.std(abs_e) / np.sqrt(n))
    rmse      = float(np.sqrt(np.mean(sq_e)))
    rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))
    mape      = float(np.mean(rel_e) * 100.0)
    mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

    if n >= 2:
        pearson_r  = float(np.corrcoef(hr_preds_all, gt_hrs_all)[0, 1])
        pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
    else:
        pearson_r, pearson_se = float("nan"), float("nan")

    mean_snr    = float(np.mean(snr_all))
    mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

    print(f"\nAggregate Metrics ({model_name}):")
    print(f"  MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
    print(f"  RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
    print(f"  MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
    print(f"  Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
    print(f"  SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

    # Export to per-model subdirectory
    model_output_dir = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(model_output_dir, exist_ok=True)

    metrics_dict = {
        "model":      model_name,
        "n_subjects": n,
        "evaluation_method": "FFT",
        "bvp_bandpass_hz":   [0.6, 3.3],
        "aggregate_metrics": {
            "MAE":     {"value": mae,      "se": mae_se,      "unit": "bpm"},
            "RMSE":    {"value": rmse,     "se": rmse_se,     "unit": "bpm"},
            "MAPE":    {"value": mape,     "se": mape_se,     "unit": "%"},
            "Pearson": {"value": pearson_r, "se": pearson_se, "unit": ""},
            "SNR":     {"value": mean_snr, "se": mean_snr_se, "unit": "dB"},
        },
        "per_subject": [
            {
                "name":               r["name"],
                "predicted_heartrate": r["predicted_heartrate"],
                "real_heartrate":      r["real_heartrate"],
                "heartrate_error":     r["heartrate_error"],
                "snr_db":              r["snr_db"],
            }
            for r in per_subject_results
        ],
    }

    json_path = os.path.join(model_output_dir, "metrics.json")
    with open(json_path, "w") as fh:
        json.dump(metrics_dict, fh, indent=2)
    print(f"\nMetrics saved to: {json_path}")

    # Export ppg_results.csv
    csv_rows = []
    for r in per_subject_results:
        csv_rows.append({
            "name":               r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "real_heartrate":      r["real_heartrate"],
            "heartrate_error":     r["heartrate_error"],
            "blink_rate":          r["blink_rate"],
        })

    results_df = pd.DataFrame(csv_rows, columns=[
        "name", "predicted_heartrate", "real_heartrate",
        "heartrate_error", "blink_rate"
    ])

    csv_path = os.path.join(model_output_dir, "ppg_results.csv")
    results_df.to_csv(csv_path, index=False)

    print(f"CSV saved to: {csv_path}")
    print()
    print(results_df.to_string(index=False))

    # Clean up model from GPU
    del model
    torch.cuda.empty_cache()

print(f"\n\nAll models processed. Results in: {OUTPUT_DIR}")


Model: PURE_PhysNet
Path:  /home/naver/disk2/HoangDPB/rPPG-Toolbox/final_model_release/PURE_PhysNet_DiffNormalized.pth
Model loaded. Total parameters: 768,577


Inference: 100%|██████████| 316/316 [01:39<00:00,  3.16it/s]



Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020', 'S021', 'S022', 'S023', 'S024', 'S025', 'S026', 'S027', 'S028', 'S029', 'S030', 'S031', 'S032', 'S033', 'S034', 'S035', 'S036', 'S037', 'S038', 'S039', 'S040', 'S041', 'S042', 'S043', 'S044', 'S045', 'S046', 'S047', 'S048', 'S049', 'S050', 'S051', 'S052', 'S053', 'S054', 'S055', 'S056', 'S057', 'S058', 'S059']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          69.434     83.537  -14.104   -7.15   27.94
S_001          67.676     80.397  -12.722   -4.86   25.42
S_002          56.250    122.479  -66.229   -9.44   30.98
S_003         100.635    114.922  -14.287  -10.29   28.36
S_004          59.326     79.125  -19.799   -6.83   29.10
S_005          68.555     92.787  -24.233   -8.21   28.41
S_006         106.78

Inference: 100%|██████████| 316/316 [00:10<00:00, 31.34it/s]



Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020', 'S021', 'S022', 'S023', 'S024', 'S025', 'S026', 'S027', 'S028', 'S029', 'S030', 'S031', 'S032', 'S033', 'S034', 'S035', 'S036', 'S037', 'S038', 'S039', 'S040', 'S041', 'S042', 'S043', 'S044', 'S045', 'S046', 'S047', 'S048', 'S049', 'S050', 'S051', 'S052', 'S053', 'S054', 'S055', 'S056', 'S057', 'S058', 'S059']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          90.527     83.537    6.990   -7.24   27.94
S_001         102.832     80.397   22.435   -8.28   25.42
S_002         152.051    122.479   29.572  -10.20   30.98
S_003         100.635    114.922  -14.287   -7.57   28.36
S_004         108.984     79.125   29.859   -7.01   29.10
S_005          97.998     92.787    5.211   -7.19   28.41
S_006         151.61

...


Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020', 'S021', 'S022', 'S023', 'S024', 'S025', 'S026', 'S027', 'S028', 'S029', 'S030', 'S031', 'S032', 'S033', 'S034', 'S035', 'S036', 'S037', 'S038', 'S039', 'S040', 'S041', 'S042', 'S043', 'S044', 'S045', 'S046', 'S047', 'S048', 'S049', 'S050', 'S051', 'S052', 'S053', 'S054', 'S055', 'S056', 'S057', 'S058', 'S059']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          82.178     83.537   -1.360   -4.79   27.94
S_001          79.102     80.397   -1.296   -1.18   25.42
S_002          84.375    122.479  -38.104   -6.15   30.98
S_003         115.137    114.922    0.215   -7.61   28.36
S_004          83.936     79.125    4.811   -4.78   29.10
S_005          85.693     92.787   -7.094   -6.20   28.41
S_006         106.34

Inference: 100%|██████████| 316/316 [00:10<00:00, 31.16it/s]



Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020', 'S021', 'S022', 'S023', 'S024', 'S025', 'S026', 'S027', 'S028', 'S029', 'S030', 'S031', 'S032', 'S033', 'S034', 'S035', 'S036', 'S037', 'S038', 'S039', 'S040', 'S041', 'S042', 'S043', 'S044', 'S045', 'S046', 'S047', 'S048', 'S049', 'S050', 'S051', 'S052', 'S053', 'S054', 'S055', 'S056', 'S057', 'S058', 'S059']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          80.420     83.537   -3.118   -3.74   27.94
S_001          79.980     80.397   -0.417    0.91   25.42
S_002          90.088    122.479  -32.391  -10.65   30.98
S_003          97.998    114.922  -16.924   -8.82   28.36
S_004          71.631     79.125   -7.494   -5.39   29.10
S_005          79.541     92.787  -13.246   -7.79   28.41
S_006         106.34

Inference: 100%|██████████| 316/316 [00:10<00:00, 31.09it/s]



Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020', 'S021', 'S022', 'S023', 'S024', 'S025', 'S026', 'S027', 'S028', 'S029', 'S030', 'S031', 'S032', 'S033', 'S034', 'S035', 'S036', 'S037', 'S038', 'S039', 'S040', 'S041', 'S042', 'S043', 'S044', 'S045', 'S046', 'S047', 'S048', 'S049', 'S050', 'S051', 'S052', 'S053', 'S054', 'S055', 'S056', 'S057', 'S058', 'S059']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          87.012     83.537    3.474   -5.44   27.94
S_001          82.178     80.397    1.780   -5.20   25.42
S_002          93.604    122.479  -28.876   -7.61   30.98
S_003          56.250    114.922  -58.672   -8.01   28.36
S_004          80.859     79.125    1.734   -6.14   29.10
S_005         101.953     92.787    9.166   -5.25   28.41
S_006         106.78